# 01. 모델 학습 및 시각화 노트북

이 노트북은 수집된 데이터를 이용해 **회귀 예측, 통폐합 위험 산정, 모델 비교, HTML 지도 생성**까지 실행하는 흐름을 정리한다.

원칙:

- 실제 로직은 `src/`의 `.py` 파일에 유지한다.
- 노트북은 실행 순서, 설명, 결과 확인을 담당한다.
- 발표 시에는 이 노트북으로 “어떤 모델을 어떤 순서로 만들었는지” 보여줄 수 있다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import HTML, IFrame, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

REPORTS = ROOT / "outputs" / "reports"
FIGURES = ROOT / "outputs" / "figures"
MAPS = ROOT / "outputs" / "maps"
PROCESSED = ROOT / "data" / "processed"

def run_script(script: str, *args: str) -> None:
    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")
    env.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
    path = ROOT / script
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"\n=== RUN {script} ===")
    subprocess.run([sys.executable, str(path), *args], cwd=ROOT, check=True, env=env)

print("project root:", ROOT)

## 1. 모델링 파이프라인 개요

최종 파이프라인은 다음 흐름이다.

1. EDSS 과거 학교ID 소멸 proxy 기반 객관 분류모델 학습
2. 학교 반경 상권/고립도 피처 생성
3. 시군구 학령인구 회귀모델 학습
4. 절대값/압력비 시나리오와 변화량 시나리오 생성
5. supervised 보조 백분위 결합
6. 출생 코호트 장기 시나리오 생성 및 최종 위험등급 반영
7. 모델 비교, 피처 중요도, 출산율 경로 분석 생성
8. 최종 인터랙티브 HTML 지도 생성

## 2. EDSS 객관 proxy 분류모델

실제 통폐합 확정 라벨이 충분하지 않기 때문에, EDSS 과거 패널에서 `학교ID가 5년 뒤 사라지는지`를 proxy 라벨로 사용한다.

이 모델은 최종 위험등급을 단독 결정하지 않고, 과거 소멸 패턴과의 유사도를 보조지표로 제공한다.

In [ ]:
# run_script("src/models/model_edss_closure_risk_national.py")

## 3. 학교 주변 상권 및 고립도 피처

학교 주변 1km 내 상권 수, 교육/아동 관련 업종, 같은 학교급까지의 거리 등을 이용해 상권 취약도와 학교 고립도를 계산한다.

In [ ]:
# run_script("src/features/build_school_radius_commercial_features.py")

## 4. 회귀모델 및 학교별 위험 시나리오 생성

이 단계에서 핵심 회귀모델이 학습된다.

- Ridge: 선형 베이스 회귀모델
- RandomForest: 튜닝 회귀모델
- 변화량 RandomForest: 인구이동과 연령구조 차이를 보기 위한 변화량 모델

변화량 모델은 `다음 해 인구 절대값`이 아니라 `다음 해 인구 증감량`을 예측한다. 장기 반복 예측의 발산을 막기 위해 연간 변화량은 `-20% ~ +10%`로 제한한다.

In [ ]:
# run_script("src/models/final_national_training_pipeline.py")

## 5. supervised 보조 백분위 결합

EDSS proxy 기반 supervised 모델의 현재 학교별 백분위를 최종 시나리오에 보조 피처로 결합한다.

In [ ]:
# run_script("src/models/final_supervised_training.py")

## 6. 출생 코호트 시나리오 적용

2023~2025년 출생아수가 초등/중등/고등 학교급으로 순차 진입한다고 보고 장기 감소를 반영한다. 최종 HTML의 기본 모델은 출생 코호트 기준이다.

In [ ]:
# run_script("src/models/build_school_level_cohort_scenario.py")
# run_script("src/models/apply_cohort_scenario_to_risk.py")

## 7. 비교 리포트 및 시각화 생성

발표용 비교 산출물을 만든다.

- Ridge vs RandomForest 회귀 성능 비교
- LogisticRegression vs HistGradientBoosting 분류 성능 비교
- 압력비/변화량/코호트 3개 모델 비교
- 타겟 설계별 피처 중요도 비교
- 출산율 경로 분석

In [ ]:
# run_script("src/reports/generate_comparison_chart.py")
# run_script("src/reports/generate_model_comparison_visuals.py")
# run_script("src/reports/extract_model_feature_importance.py")
# run_script("src/reports/fertility_pathway_analysis.py")
# run_script("src/reports/generate_final_reports_and_visuals.py")

## 8. 최종 HTML 지도 생성

최종 발표/시연용 지도는 `outputs/maps/final_national_interactive_school_risk_scenario.html`이다.

지도에서 확인할 수 있는 것:

- 2026~2040 연도 선택
- 지역/학교급/위험등급 필터
- 출생 코호트 기준, 변화량 모델, 압력비 모델 선택
- 위험 피처 체크박스 시뮬레이션
- 그래프 탭의 3개 예측 모델 비교

In [ ]:
# run_script("src/viz/build_final_interactive_school_risk_map.py")

## 9. 성능지표 확인

이미 생성된 CSV를 읽어 주요 성능지표를 확인한다.

In [ ]:
reg_path = REPORTS / "regression_base_vs_tuned_comparison.csv"
clf_path = REPORTS / "classification_base_vs_tuned_comparison.csv"

if reg_path.exists():
    display(pd.read_csv(reg_path))
else:
    print("missing:", reg_path)

if clf_path.exists():
    display(pd.read_csv(clf_path))
else:
    print("missing:", clf_path)

## 10. 3개 예측 모델 비교 요약

압력비, 변화량, 출생 코호트 모델의 2040 예측 차이를 확인한다.

In [ ]:
summary_path = REPORTS / "model_regional_differentiation_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("missing:", summary_path)

## 11. 발표용 그림 확인

생성된 주요 그림 파일 경로를 확인한다.

In [ ]:
figure_names = [
    "regression_base_vs_tuned_comparison.png",
    "classification_base_vs_tuned_comparison.png",
    "three_model_comparison.png",
    "sido_decrease_heatmap_by_model.png",
    "target_design_feature_comparison.png",
    "migration_effect_comparison.png",
]
for name in figure_names:
    path = FIGURES / name
    print(f"{name}: {path.exists()} | {path}")

## 12. 최종 지도 열기

Jupyter 환경에서 HTML iframe으로 확인한다. 브라우저에서 직접 열 때는 아래 파일을 열면 된다.

`outputs/maps/final_national_interactive_school_risk_scenario.html`

In [ ]:
map_path = MAPS / "final_national_interactive_school_risk_scenario.html"
print(map_path)
if map_path.exists():
    display(IFrame(src=str(map_path), width="100%", height=720))
else:
    print("missing:", map_path)